# Heart Disease Prediction
The core Purpose is to get fimiliar with Classification algorithms and to predict the presence of heart disease in patients based on various features.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [24]:
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier


In [18]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [15]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    accuracy_score,
)

### Loading the Data

In [4]:
# Loading the Data 
df = pd.read_csv('heart_disease_uci.csv')
df.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


## Understanding the Data

In [7]:
# Shape (rows, columns)
print(f"Dataset has {df.shape[0]} Patient's data and {df.shape[1]} features")

# Column names
print("\nColumns:", df.columns.tolist())

# Data types
print("\nData types:")
print(df.dtypes)

# Summary statistics
df.describe()

Dataset has 920 Patient's data and 16 features

Columns: ['id', 'age', 'sex', 'dataset', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']

Data types:
id            int64
age           int64
sex          object
dataset      object
cp           object
trestbps    float64
chol        float64
fbs          object
restecg      object
thalch      float64
exang        object
oldpeak     float64
slope        object
ca          float64
thal         object
num           int64
dtype: object


,id,age,trestbps,chol,thalch,oldpeak,ca,num
count,920.000000,920.000000,861.000000,890.000000,865.000000,858.000000,309.000000,920.000000
mean,460.500000,53.510870,132.132404,199.130337,137.545665,0.878788,0.676375,0.995652
std,265.725422,9.424685,19.066070,110.780810,25.926276,1.091226,0.935653,1.142693
min,1.000000,28.000000,0.000000,0.000000,60.000000,-2.600000,0.000000,0.000000
25%,230.750000,47.000000,120.000000,175.000000,120.000000,0.000000,0.000000,0.000000
50%,460.500000,54.000000,130.000000,223.000000,140.000000,0.500000,0.000000,1.000000
75%,690.250000,60.000000,140.000000,268.000000,157.000000,1.500000,1.000000,2.000000
max,920.000000,77.000000,200.000000,603.000000,202.000000,6.200000,3.000000,4.000000


## Checking for Missing Values

In [8]:
# Count missing values
missing = df.isnull().sum() # Here df is the DataFrame we are working with
print(missing[missing > 0])

# Percentage missing
missing_percent = (df.isnull().sum() / len(df)) * 100
print("\nPercentage missing:")
print(missing_percent[missing_percent > 0])

trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
dtype: int64

Percentage missing:
trestbps     6.413043
chol         3.260870
fbs          9.782609
restecg      0.217391
thalch       5.978261
exang        5.978261
oldpeak      6.739130
slope       33.586957
ca          66.413043
thal        52.826087
dtype: float64


## Filling Missing Values

In [ ]:
# Dropping the ca due to it missing alot of values. Also dropping thal due to it missing alot of values.
df = df.drop(columns=['ca'])
df = df.drop(columns=['thal']) # Now the Columns with the most missing values have been dealt with. 

 Data Cleaning & Missing-Value Handling (what we did and why)

### 1.1 Create a **binary** target from `num`
```python
df["target"] = (df["num"].fillna(0) != 0).astype(int)
```

**Meaning (in plain words):**
- In this dataset, `num` represents the diagnosis.
  - Usually `num = 0` means **no heart disease**
  - `num = 1,2,3,4` means **heart disease is present** (different severity levels)
- Our goal is **binary classification** (YES/NO), so we convert it to:
  - `target = 0` if `num == 0`
  - `target = 1` if `num != 0`

**Why `fillna(0)`?**
- If `num` has missing values, we temporarily treat them as `0` so the conversion doesn’t crash.
- (In a strict medical pipeline, we’d also investigate why `num` is missing, but for learning and baseline modeling, this is OK.)

---

### 1.2 Drop `id`
```python
df = df.drop(columns=["id"], errors="ignore")
```

**Meaning:**
- `id` is just a **unique identifier** for each patient.
- It doesn’t describe health and has no real predictive value.
- Keeping it can confuse the model because the model might try to learn patterns from random ID numbers.

---

### 1.3 Fill missing **categorical** columns with `"Unknown"`
```python
categorical_cols = ["slope", "restecg", "cp", "sex", "origin"]
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype("object").fillna("Unknown")
```

**Meaning:**
- These columns store *labels* like `Male/Female`, chest pain types, ECG categories, etc.
- If the value is missing, we do **not guess** a category.
- We honestly label it `"Unknown"` meaning: *“this was not recorded.”*

**Why this is useful in health data:**
- Sometimes **missing itself is information** (maybe a test wasn’t performed).
- `"Unknown"` keeps the patient row in the dataset without inventing medical information.

---

### 1.4 Treat True/False columns (`fbs`, `exang`) like categories and fill `"Unknown"`
```python
bool_like_cols = ["fbs", "exang"]
for col in bool_like_cols:
    if col in df.columns:
        df[col] = df[col].astype("object").fillna("Unknown")
```

**Meaning:**
- `fbs` and `exang` are often True/False.
- If missing, we again avoid guessing True or False.
- We use `"Unknown"`.

---

### 1.5 Fill missing **numeric** columns with the **median**
```python
numeric_fill_cols = ["trestbps", "chol", "thalach", "thalch", "oldpeak", "age"]
for col in numeric_fill_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())
```

**Meaning:**
- These columns are real numbers: blood pressure, cholesterol, heart rate, etc.
- We replace missing values with the **median** (the “middle” value).

**Why median (instead of mean/average)?**
- Medical measurements often have extreme values (outliers).
- The median is more “stable” and less affected by very high/low values.

**Why `pd.to_numeric(..., errors="coerce")`?**
- It forces the column to be numeric.
- Any weird values become missing (`NaN`) so they can be handled safely.

---

In [10]:
# 1) Identify target and create binary target from `num`
#    (Keep `num` if you want to inspect later; but create `target` for modeling)
df["target"] = (df["num"].fillna(0) != 0).astype(int)

# Optional: drop id from features (good practice)
df = df.drop(columns=["id"], errors="ignore")

# 2) Fill categorical columns with "Unknown"
categorical_cols = ["slope", "restecg", "cp", "sex", "origin"]
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype("object").fillna("Unknown")

# 3) Fill True/False style columns with "Unknown" (treat as categorical)
bool_like_cols = ["fbs", "exang"]
for col in bool_like_cols:
    if col in df.columns:
        df[col] = df[col].astype("object").fillna("Unknown")

# 4) Fill numeric columns with median (safer than mean)
numeric_fill_cols = ["trestbps", "chol", "thalach", "thalch", "oldpeak", "age"]
for col in numeric_fill_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")  # ensures numeric; invalid -> NaN
        df[col] = df[col].fillna(df[col].median())

In [13]:
# Count missing values
missing = df.isnull().sum() # Here df is the DataFrame we are working with
print(missing[missing > 0]) # Should show no missing values now

Series([], dtype: int64)


# Model training and evaluation will be done in the next steps.

In [16]:
# Train/test split
from sklearn.model_selection import train_test_split

# 1) Target column (binary)
target = "target"  # we created this from `num` earlier

# 2) Features (everything except target and raw num)
X = df.drop(columns=[target, "num", "id"], errors="ignore")
y = df[target]

# 3) Split (keep class ratio similar in train/test)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 2) Evaluation Function (what it prints and why)

We used an evaluation helper function so we can score *any* classifier in the same way.

### What the function does:
1. **Predicts classes** (0/1) on the test set
2. Prints **Confusion Matrix** (what kinds of mistakes we made)
3. Prints **Classification Report**:
   - precision
   - recall
   - f1-score
4. Computes **ROC-AUC** and **PR-AUC** (how well the model ranks risk)
5. Prints **Accuracy** (but we don’t rely on it alone)

### Why this is better than RMSE/MAE:
- RMSE/MAE are for predicting a number (regression).
- Here we’re predicting **YES/NO**, so classification metrics are the correct tool.

---

In [17]:
# Evaluation helper for BINARY CLASSIFICATION models
def eval_classifier(model, X_test, y_test, name="model"):
    """
    Prints a small, consistent evaluation summary for a binary classifier.

    Assumptions:
    - model implements .predict(X) -> class labels (0/1)
    - model implements either .predict_proba(X) or .decision_function(X) for probability-like scoring
    """
    y_pred = model.predict(X_test)

    print(f"\n{name}")
    print("-" * len(name))

    # 1) The "counts table" of correct/wrong predictions
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    # 2) Accuracy, Precision, Recall, F1 in one report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=3))

    # 3) ROC-AUC and PR-AUC need scores (probabilities or decision scores)
    y_score = None
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)

    if y_score is not None:
        print("ROC-AUC:", round(roc_auc_score(y_test, y_score), 4))
        print("PR-AUC :", round(average_precision_score(y_test, y_score), 4))
    else:
        print("ROC-AUC: (skipped - model has no predict_proba/decision_function)")
        print("PR-AUC : (skipped - model has no predict_proba/decision_function)")

    # 4) Plain accuracy (sometimes useful, but don't trust it alone for imbalance)
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))

## Creating the Pre-process Blocks 


 Why we still needed a `preprocess` block even after cleaning missing values

Even after filling missing values, we still have columns like:
- `sex` (Male/Female)
- `cp` (types of pain)
- `restecg` (categories)
- `"Unknown"` labels

**Logistic Regression and many ML models cannot directly learn from words.**

So the preprocess block does two jobs:
1. **Numeric columns**: (optional) scale them so they’re on similar ranges
2. **Categorical columns**: convert categories into numeric columns (one-hot encoding)

In short:
- Cleaning fixes missing values
- Preprocessing makes the data *machine-readable* for the model

---

In [19]:
# X is your features dataframe (NOT including target/num)
# Example:
# X = df.drop(columns=["target", "num", "id"], errors="ignore")

# 1) Automatically pick which columns are numeric vs categorical
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# 2) Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),  # will keep "Unknown" if it's the most frequent
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

# 3) Combine them
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

## 4) Why we converted `fbs` and `exang` to strings

```python
for col in ["fbs", "exang"]:
    X[col] = X[col].astype("object").astype(str)
```

**The problem we hit:**
- These columns had mixed values like:
  - `True`, `False` (boolean)
  - `"Unknown"` (string)

That mixture causes the encoder to crash because it can’t sort/compare booleans and strings.

**The fix:**
- Convert everything to the same type (strings):
  - `"True"`, `"False"`, `"Unknown"`

This makes one-hot encoding work reliably.

---

In [21]:
# Columns that are True/False but also have "Unknown"
for col in ["fbs", "exang"]:
    if col in X.columns:
        X[col] = X[col].astype("object").astype(str)

# If you already made X_train/X_test, apply to them too:
for col in ["fbs", "exang"]:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype("object").astype(str)
        X_test[col] = X_test[col].astype("object").astype(str)

# Logistic Regression (plain explanation + example)

### What it is
Logistic Regression is a **binary decision model**:
- It outputs a probability between 0 and 1
- Then it turns that into a YES/NO decision using a cutoff (often 0.5)

### Example
- Patient A → model outputs 0.82 → predict **Disease**
- Patient B → model outputs 0.31 → predict **No Disease**

### Why it’s strong as a baseline
- Simple, fast, stable
- Often performs very well
- Provides probabilities, which is useful for risk scoring in healthcare

---

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Baseline model using Logistic Regression (binary classification)
logreg = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

logreg.fit(X_train, y_train)

# Call the evaluation function we created earlier
eval_classifier(logreg, X_test, y_test, name="Logistic Regression")


Logistic Regression
-------------------
Confusion Matrix:
[[64 18]
 [15 87]]

Classification Report:
              precision    recall  f1-score   support

           0      0.810     0.780     0.795        82
           1      0.829     0.853     0.841       102

    accuracy                          0.821       184
   macro avg      0.819     0.817     0.818       184
weighted avg      0.820     0.821     0.820       184

ROC-AUC: 0.92
PR-AUC : 0.9221
Accuracy: 0.8207


# Decision Tree Classifier (plain explanation + example)

### What it is
A Decision Tree is like a chain of “if-else” questions.

### Example logic (conceptual)
- If chest pain is “asymptomatic” → higher risk
- If max heart rate is low and oldpeak is high → higher risk
- The tree keeps splitting until it decides:
  - Disease / No disease

### Pros
- Easy to explain
- Captures non-linear patterns (complex rule combinations)

### Cons
- Can overfit (memorize the training data)
- Sensitive to tree depth and settings

That’s why we tuned `max_depth`.

---

In [26]:
# Now running the Decision Tree Classifier (This is the Tuneable Model)
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

depths = [2, 3, 4, 5, 6, 8, None]

for d in depths:
    tree = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", DecisionTreeClassifier(
            random_state=42,
            class_weight="balanced",
            max_depth=d
        ))
    ])

    tree.fit(X_train, y_train)
    eval_classifier(tree, X_test, y_test, name=f"Decision Tree (max_depth={d})")


Decision Tree (max_depth=2)
---------------------------
Confusion Matrix:
[[60 22]
 [12 90]]

Classification Report:
              precision    recall  f1-score   support

           0      0.833     0.732     0.779        82
           1      0.804     0.882     0.841       102

    accuracy                          0.815       184
   macro avg      0.818     0.807     0.810       184
weighted avg      0.817     0.815     0.814       184

ROC-AUC: 0.8574
PR-AUC : 0.8528
Accuracy: 0.8152

Decision Tree (max_depth=3)
---------------------------
Confusion Matrix:
[[74  8]
 [28 74]]

Classification Report:
              precision    recall  f1-score   support

           0      0.725     0.902     0.804        82
           1      0.902     0.725     0.804       102

    accuracy                          0.804       184
   macro avg      0.814     0.814     0.804       184
weighted avg      0.824     0.804     0.804       184

ROC-AUC: 0.8924
PR-AUC : 0.8777
Accuracy: 0.8043

Decision Tr

In [27]:
def eval_compact(model, X_test, y_test):
    y_pred = model.predict(X_test)

    # get scores for AUCs
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)

    return {
        "accuracy": report["accuracy"],
        "recall_1": report["1"]["recall"],
        "precision_1": report["1"]["precision"],
        "f1_1": report["1"]["f1-score"],
        "fn": int(cm[1, 0]),
        "fp": int(cm[0, 1]),
        "roc_auc": roc_auc_score(y_test, y_score),
        "pr_auc": average_precision_score(y_test, y_score),
    }

depths = [1, 2, 3, 4, 5, 6, 8, None]
results = []

for d in depths:
    tree = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", DecisionTreeClassifier(
            random_state=42,
            class_weight="balanced",
            max_depth=d
        ))
    ])
    tree.fit(X_train, y_train)
    r = eval_compact(tree, X_test, y_test)
    r["max_depth"] = d
    results.append(r)

# Print sorted by "recall_1" (catching sick patients), then by PR-AUC
results_sorted = sorted(results, key=lambda x: (x["recall_1"], x["pr_auc"]), reverse=True)

for r in results_sorted:
    print(
        f"depth={r['max_depth']}: "
        f"recall_1={r['recall_1']:.3f}, precision_1={r['precision_1']:.3f}, "
        f"FN={r['fn']}, FP={r['fp']}, "
        f"PR-AUC={r['pr_auc']:.3f}, ROC-AUC={r['roc_auc']:.3f}, "
        f"acc={r['accuracy']:.3f}"
    )

depth=2: recall_1=0.882, precision_1=0.804, FN=12, FP=22, PR-AUC=0.853, ROC-AUC=0.857, acc=0.815
depth=1: recall_1=0.833, precision_1=0.825, FN=17, FP=18, PR-AUC=0.780, ROC-AUC=0.807, acc=0.810
depth=4: recall_1=0.804, precision_1=0.828, FN=20, FP=17, PR-AUC=0.849, ROC-AUC=0.876, acc=0.799
depth=6: recall_1=0.794, precision_1=0.862, FN=21, FP=13, PR-AUC=0.782, ROC-AUC=0.825, acc=0.815
depth=None: recall_1=0.765, precision_1=0.772, FN=24, FP=23, PR-AUC=0.721, ROC-AUC=0.742, acc=0.745
depth=3: recall_1=0.725, precision_1=0.902, FN=28, FP=8, PR-AUC=0.878, ROC-AUC=0.892, acc=0.804
depth=8: recall_1=0.725, precision_1=0.813, FN=28, FP=17, PR-AUC=0.774, ROC-AUC=0.769, acc=0.755
depth=5: recall_1=0.667, precision_1=0.872, FN=34, FP=10, PR-AUC=0.835, ROC-AUC=0.857, acc=0.761


# Overall Report 
 Report: Comparing Logistic Regression vs Decision Tree (best tuned depth)

### Logistic Regression (baseline)
Confusion Matrix:
- TN=64, FP=18
- FN=15, TP=87

Key metrics:
- Class 1 (Disease): precision **0.829**, recall **0.853**, F1 **0.841**
- ROC-AUC **0.920**
- PR-AUC **0.922**
- Accuracy **0.821**

**Interpretation:**
- Strong overall model.
- Good at ranking high-risk patients (high ROC-AUC and PR-AUC).
- Missed 15 disease cases (FN=15).

---

### Decision Tree (best: max_depth=2)
Confusion Matrix:
- TN=60, FP=22
- FN=12, TP=90

Key metrics:
- Class 1 (Disease): precision **0.804**, recall **0.882**, F1 **0.841**
- ROC-AUC **0.857**
- PR-AUC **0.853**
- Accuracy **0.815**

**Interpretation:**
- Catches slightly more disease cases than logistic regression (FN=12 vs 15).
- But gives more false alarms (FP=22 vs 18).
- Worse overall ranking power than logistic regression (lower ROC-AUC and PR-AUC).

---

### Final comparison (which is “better” and why)
- **If the top goal is to miss as few sick patients as possible (higher recall):**
  - Decision Tree (depth=2) is slightly better (recall 0.882 vs 0.853).
- **If the goal is the strongest overall model and best risk-ranking ability:**
  - Logistic Regression is better (ROC-AUC 0.92 and PR-AUC 0.922 vs ~0.85).

**Conclusion:**
- Logistic Regression is the best **overall** baseline on this dataset.
- The tuned Decision Tree (depth=2) is a valid alternative if the priority is maximizing disease recall, at the cost of more false alarms.

---

## Saving the Best Model

In [29]:
import joblib
# Save the best model (based on evaluation metrics)
best_tree = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced",
        max_depth=2  # replace with the best depth you found
    ))
])
best_tree.fit(X_train, y_train)
joblib.dump(best_tree, "best_decision_tree_depth2.joblib")

['best_decision_tree_depth2.joblib']